# 02 — what extends?

**triplet-phase-lock**

This notebook moves from raw candidate generation to local continuation. We compare neighboring triplets, measure how structure extends across steps, and contrast a clean sequence with a perturbed variant.

Guiding question:

- **π (extend): what extends?**


## Setup

We reuse the base sequence from Notebook 01:

\\[\nN_k = 24k - 25\n\\]\n
and package local structure into triplets:

\\[\nT_k = (N_k, N_{k+1}, N_{k+2})\n\\]\n
To study extension, we compare each triplet to its neighbor using:

\\[\n\Delta_k = \\|T_{k+1} - T_k\\|\n\\]\n
We also compare against a perturbed sequence to see what continues coherently and what drifts.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


## Sequence and triplet helpers


In [ ]:
def sequence_n(k):
    """
    Base sequence:
        N_k = 24k - 25
    """
    k = np.asarray(k, dtype=float)
    return 24.0 * k - 25.0


def sequence_n_perturbed(k, amplitude=8.0, period=6.0):
    """
    Perturbed comparison sequence:
        N_k + amplitude * sin(k / period)
    """
    k = np.asarray(k, dtype=float)
    return sequence_n(k) + amplitude * np.sin(k / period)


def build_triplets_from_values(vals):
    """
    Build triplets from an array of values.
    """
    vals = np.asarray(vals, dtype=float)
    return np.stack([vals[:-2], vals[1:-1], vals[2:]], axis=1)


def build_triplets(num_steps, sequence_fn):
    ks = np.arange(1, num_steps + 3, dtype=float)
    vals = sequence_fn(ks)
    return build_triplets_from_values(vals)


def local_differences(x):
    """
    Euclidean norm of neighboring triplet differences.
    """
    if len(x) < 2:
        return np.array([], dtype=float)
    return np.linalg.norm(np.diff(x, axis=0), axis=1)


def component_differences(x):
    """
    Neighboring triplet differences, componentwise.
    """
    if len(x) < 2:
        return np.empty((0, x.shape[1]), dtype=float)
    return np.diff(x, axis=0)


def normalize_rows(x, eps=1e-12):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


## Generate clean and perturbed triplets


In [ ]:
num_steps = 60
k = np.arange(1, num_steps + 3)

clean_vals = sequence_n(k)
perturbed_vals = sequence_n_perturbed(k, amplitude=8.0, period=6.0)

triplets_clean = build_triplets_from_values(clean_vals)
triplets_perturbed = build_triplets_from_values(perturbed_vals)

print('First 5 clean triplets:')
print(np.round(triplets_clean[:5], 2))

print('\nFirst 5 perturbed triplets:')
print(np.round(triplets_perturbed[:5], 2))


## Plot 1 — clean vs perturbed sequence

This gives the first visual sense of what may continue coherently and what may drift.


In [ ]:
plt.figure()
plt.plot(k, clean_vals, marker='o', label='clean sequence')
plt.plot(k, perturbed_vals, marker='o', label='perturbed sequence')
plt.xlabel('k')
plt.ylabel('value')
plt.title('Sequence comparison: clean vs perturbed')
plt.legend()
plt.tight_layout()
plt.show()


## Local triplet differences

Measure neighboring triplet changes:

\\[\n\Delta_k = \\|T_{k+1} - T_k\\|\n\\]\n
For the clean sequence, this should be constant. For the perturbed sequence, it should vary.


In [ ]:
delta_clean = local_differences(triplets_clean)
delta_perturbed = local_differences(triplets_perturbed)

print('Unique clean Δ values (rounded):', np.unique(np.round(delta_clean, 6)))
print('Perturbed Δ range:', np.round(delta_perturbed.min(), 4), 'to', np.round(delta_perturbed.max(), 4))


In [ ]:
idx = np.arange(1, len(delta_clean) + 1)

plt.figure()
plt.plot(idx, delta_clean, marker='o', label='clean Δ_k')
plt.plot(idx, delta_perturbed, marker='o', label='perturbed Δ_k')
plt.xlabel('k')
plt.ylabel(r'$\Delta_k$')
plt.title('Local triplet extension magnitude')
plt.legend()
plt.tight_layout()
plt.show()


## Componentwise neighboring differences

Inspect the vector difference:

\\[\nT_{k+1} - T_k\n\\]\n
This shows whether extension remains uniform across coordinates.


In [ ]:
diffs_clean = component_differences(triplets_clean)
diffs_perturbed = component_differences(triplets_perturbed)

print('First 5 clean component differences:')
print(np.round(diffs_clean[:5], 2))

print('\nFirst 5 perturbed component differences:')
print(np.round(diffs_perturbed[:5], 2))


In [ ]:
comp_idx = np.arange(1, len(diffs_clean) + 1)

plt.figure()
plt.plot(comp_idx, diffs_clean[:, 0], label='clean: ΔN_k')
plt.plot(comp_idx, diffs_clean[:, 1], label='clean: ΔN_{k+1}')
plt.plot(comp_idx, diffs_clean[:, 2], label='clean: ΔN_{k+2}')
plt.xlabel('k')
plt.ylabel('component difference')
plt.title('Clean triplet component differences')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
plt.plot(comp_idx, diffs_perturbed[:, 0], label='perturbed: ΔN_k')
plt.plot(comp_idx, diffs_perturbed[:, 1], label='perturbed: ΔN_{k+1}')
plt.plot(comp_idx, diffs_perturbed[:, 2], label='perturbed: ΔN_{k+2}')
plt.xlabel('k')
plt.ylabel('component difference')
plt.title('Perturbed triplet component differences')
plt.legend()
plt.tight_layout()
plt.show()


## Direction vectors

Normalize neighboring triplet differences to inspect local direction rather than only magnitude.

This helps distinguish coherent extension from local oscillatory drift.


In [ ]:
dirs_clean = normalize_rows(diffs_clean)
dirs_perturbed = normalize_rows(diffs_perturbed)

print('First 5 clean normalized directions:')
print(np.round(dirs_clean[:5], 4))

print('\nFirst 5 perturbed normalized directions:')
print(np.round(dirs_perturbed[:5], 4))


In [ ]:
plt.figure()
plt.plot(comp_idx, dirs_clean[:, 0], label='clean dir x')
plt.plot(comp_idx, dirs_clean[:, 1], label='clean dir y')
plt.plot(comp_idx, dirs_clean[:, 2], label='clean dir z')
plt.xlabel('k')
plt.ylabel('normalized component')
plt.title('Normalized direction: clean triplets')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
plt.plot(comp_idx, dirs_perturbed[:, 0], label='perturbed dir x')
plt.plot(comp_idx, dirs_perturbed[:, 1], label='perturbed dir y')
plt.plot(comp_idx, dirs_perturbed[:, 2], label='perturbed dir z')
plt.xlabel('k')
plt.ylabel('normalized component')
plt.title('Normalized direction: perturbed triplets')
plt.legend()
plt.tight_layout()
plt.show()


## 2D triplet-path comparison

Project each triplet path into the first two coordinates.

The clean case should remain tightly ordered, while the perturbed case bends away from a single straight-line path.


In [ ]:
plt.figure()
plt.plot(triplets_clean[:, 0], triplets_clean[:, 1], marker='o', label='clean path')
plt.plot(triplets_perturbed[:, 0], triplets_perturbed[:, 1], marker='o', label='perturbed path')
plt.xlabel(r'$N_k$')
plt.ylabel(r'$N_{k+1}$')
plt.title('2D triplet-path comparison')
plt.legend()
plt.tight_layout()
plt.show()


## Summary

This notebook established the **extend** stage:

- compared neighboring triplets
- measured local extension magnitude
- inspected componentwise continuation
- contrasted clean vs perturbed trajectories

Next notebook:

- **Π (resist): what resists?**
- introduce cosine alignment
- apply the 45° threshold
- measure resistance scores and bounded alignment
